# PyBullet

البته! در زیر کل محتوای آموزشی **PyBullet** را به صورت یک فایل واحد و کامل در **قالب Markdown** (`.md`) آورده‌ام که می‌توانید آن را مستقیماً کپی کنید و در یک فایل با نام مثلاً `pybullet_tutorial.md` ذخیره کنید.

این فایل شامل تمام بخش‌هاست:  
✅ مقدمه  
✅ نصب  
✅ مثال‌های اجرایی  
✅ توضیحات داخلی  
✅ تمرین‌ها  
✅ رفع خطا  
✅ پروژه‌های پیشنهادی  
✅ استفاده در Colab  
✅ منابع

---

```markdown
# 📘 آموزش جامع و کاربردی PyBullet  
## شبیه‌سازی ربات، فیزیک و محیط‌های تعاملی با Python  
**برای دانشجویان رشته‌های رباتیک، هوش مصنوعی و سیستم‌های کنترل**

---

## 📌 چه چیزی یاد خواهیم گرفت؟

در این آموزش، با کتابخانه **PyBullet** آشنا می‌شویم — یک ابزار قدرتمند و رایگان برای شبیه‌سازی فیزیک، رباتیک و محیط‌های تعاملی. این ابزار به شما کمک می‌کند تا:

- ربات‌ها را در محیط سه‌بعدی شبیه‌سازی کنید.
- کنترلرهای حرکتی (مانند PID) پیاده‌سازی کنید.
- از حسگرهای مختلف (دوربین، تماس، موقعیت جوینت) استفاده کنید.
- پروژه‌های یادگیری تقویتی (Reinforcement Learning) را تست کنید.

> ✅ این آموزش برای دانشجویان سطح متوسط به بالا طراحی شده است. نیاز به دانش پایه‌ای از Python، رباتیک و فیزیک دارد.

---

## 🔧 نصب و راه‌اندازی

ابتدا کتابخانه `pybullet` را نصب کنید:

```bash
pip install pybullet
```

> ⚠️ نکته: اگر از Google Colab استفاده می‌کنید، حالت `p.GUI` کار نمی‌کند. برای نمایش تصویر، باید از `pyvirtualdisplay` استفاده کنید. در ادامه راهنمایی لازم ارائه شده است.

---

## 🧪 بخش ۱: اولین شبیه‌سازی — ایجاد محیط پایه

هدف: ایجاد یک محیط ساده با گرانش، زمین و یک ربات.

### 🔹 کد اجراشدنی

```python
import pybullet as p
import pybullet_data
import time

# اتصال به شبیه‌سازی (حالت گرافیکی)
p.connect(p.GUI)

# تنظیم مسیر فایل‌های پیش‌فرض (مثل plane.urdf)
p.setAdditionalSearchPath(pybullet_data.getDataPath())

# تنظیم گرانش زمین (m/s²)
p.setGravity(0, 0, -9.81)

# بارگذاری زمین
planeId = p.loadURDF("plane.urdf")

# بارگذاری ربات KUKA IIWA
robotId = p.loadURDF("kuka_iiwa/model.urdf", [0, 0, 0])

# اجرای شبیه‌سازی به مدت ۵ ثانیه
for i in range(500):
    p.stepSimulation()
    time.sleep(1./240.)  # شبیه‌سازی زمان واقعی (240 فریم بر ثانیه)

# قطع ارتباط با شبیه‌سازی
p.disconnect()
```

### 📝 توضیحات

- `p.connect(p.GUI)`: محیط گرافیکی را باز می‌کند.
- `pybullet_data.getDataPath()`: مسیر فایل‌های از پیش تعریف شده (مثل ربات و زمین) را برمی‌گرداند.
- `p.loadURDF(...)`: یک مدل فیزیکی را بر اساس فرمت URDF (Unified Robot Description Format) بارگذاری می‌کند.
- `p.stepSimulation()`: یک گام فیزیکی شبیه‌سازی را اجرا می‌کند.

> 💡 تمرین ۱: اگر `useFixedBase=False` باشد، چه اتفاقی می‌افتد؟ ربات ممکن است بیفتد یا حرکت کند.

---

## 🎮 بخش ۲: کنترل ربات — حرکت به سمت هدف

هدف: حرکت دادن بازوی ربات به یک حالت مشخص با کنترل موقعیت.

### 🔹 کد اجراشدنی

```python
import pybullet as p
import pybullet_data
import time

p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.setGravity(0, 0, -9.81)
p.loadURDF("plane.urdf")

# بارگذاری ربات با پایه ثابت
robotId = p.loadURDF("kuka_iiwa/model.urdf", [0, 0, 0], useFixedBase=True)

# تعداد جوینت‌ها
numJoints = p.getNumJoints(robotId)
print(f"✅ تعداد جوینت‌ها: {numJoints}")

# حالت هدف (رادیان)
targetPos = [0, 0.7, 0, -1.5, 0, 1.0, 0]

# ریست کردن حالت اولیه
for i in range(7):
    p.resetJointState(robotId, i, targetPos[i])

# کنترل موقعیت با کنترل‌کننده PID داخلی
for i in range(1000):
    for j in range(7):
        p.setJointMotorControl2(
            bodyIndex=robotId,
            jointIndex=j,
            controlMode=p.POSITION_CONTROL,
            targetPosition=targetPos[j],
            force=500  # حداکثر نیروی مجاز
        )
    p.stepSimulation()
    time.sleep(1./240.)
```

### 📝 توضیحات

- `useFixedBase=True`: جلوی حرکت پایه ربات را می‌گیرد (مهم برای ربات‌های بازویی).
- `setJointMotorControl2`: نوع کنترل (مثلاً موقعیت، سرعت، نیرو) و پارامترهای آن را مشخص می‌کند.
- `force`: حداکثر گشتاوری که موتور می‌تواند اعمال کند.

> 💡 تمرین ۲: یک حالت هدف جدید انتخاب کنید (مثلاً `[0.5, 0.5, -0.5, -1, 0.3, 0.8, 0.2]`) و ربات را به آن حرکت دهید.

---

## 🧱 بخش ۳: اضافه کردن اشیا و بررسی تعامل

هدف: اضافه کردن یک جعبه و تشخیص برخورد با ربات.

### 🔹 کد اجراشدنی

```python
import pybullet as p
import pybullet_data
import time

p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.setGravity(0, 0, -9.81)
p.loadURDF("plane.urdf")
robotId = p.loadURDF("kuka_iiwa/model.urdf", [0, 0, 0], useFixedBase=True)

# اضافه کردن جعبه قرمز در مسیر بازو
boxId = p.loadURDF("cube.urdf", [0.5, 0, 0.5], globalScaling=0.3)
p.changeVisualShape(boxId, -1, rgbaColor=[1, 0, 0, 1])  # رنگ قرمز

# شبیه‌سازی و بررسی برخورد
for i in range(1000):
    p.stepSimulation()
    
    # بررسی تماس بین ربات و جعبه
    contacts = p.getContactPoints(bodyA=robotId, bodyB=boxId)
    if len(contacts) > 0:
        print(f"⚠️ برخورد در فریم {i}! نیرو: {contacts[0][9]:.2f} نیوتن")
    
    time.sleep(1./240.)
```

### 📝 توضیحات

- `globalScaling`: اندازه شیء را تغییر می‌دهد (مثلاً 0.3 = 30% اندازه اصلی).
- `rgbaColor`: رنگ شیء را تنظیم می‌کند (Red, Green, Blue, Alpha).
- `getContactPoints`: لیستی از نقاط تماس بین دو جسم را برمی‌گرداند.

> 💡 تمرین ۳: موقعیت جعبه را تغییر دهید تا فقط در یک حالت خاص با ربات برخورد کند.

---

## 📊 بخش ۴: دریافت داده‌های حسگری

### ۴.۱. خواندن وضعیت جوینت‌ها

```python
joint_states = p.getJointStates(robotId, range(7))
for i, state in enumerate(joint_states):
    pos, vel, reaction_forces, applied_torque = state
    print(f"🔸 جوینت {i}: موقعیت={pos:.3f}, سرعت={vel:.3f}, گشتاور={applied_torque:.3f}")
```

> ✅ این داده‌ها برای کنترل فیدبک (مثل PID) بسیار مفید هستند.

---

### ۴.۲. دریافت تصویر از دوربین شبیه‌سازی

هدف: گرفتن تصویر از دید یک دوربین مجازی.

```python
# تنظیم دوربین
width, height = 640, 480
view_matrix = p.computeViewMatrix(
    cameraEyePosition=[0.7, 0, 0.7],     # موقعیت دوربین
    cameraTargetPosition=[0.3, 0, 0.3],  # نقطه هدف
    cameraUpVector=[0, 0, 1]             # جهت بالا
)
proj_matrix = p.computeProjectionMatrixFOV(
    fov=60, aspect=width/height, nearVal=0.01, farVal=100
)

# گرفتن تصویر
img = p.getCameraImage(width, height, view_matrix, proj_matrix)
rgb_img = img[2]  # تصویر رنگی (RGB)
depth_img = img[3]  # نقشه عمق
seg_img = img[4]   # نقشه تفکیک اجسام

print("ابعاد تصویر:", rgb_img.shape)  # (480, 640, 3)
```

> 💡 برای نمایش تصویر از `matplotlib` استفاده کنید:

```python
import matplotlib.pyplot as plt
plt.imshow(rgb_img)
plt.title("تصویر شبیه‌سازی‌شده از دوربین")
plt.axis("off")
plt.show()
```

> 💡 تمرین ۴: تصویر و نقشه عمق را کنار هم نمایش دهید.

---

## 🎥 بخش ۵: ضبط ویدیو و لاگ داده

### ۵.۱. ضبط ویدیو از شبیه‌سازی

```python
# شروع ضبط
video_id = p.startStateLogging(p.STATE_LOGGING_VIDEO_MP4, "robot_demo.mp4")

# شبیه‌سازی چند ثانیه
for i in range(500):
    p.stepSimulation()
    time.sleep(1./240.)

# پایان ضبط
p.stopStateLogging(video_id)
print("✅ ویدیو ذخیره شد: robot_demo.mp4")
```

> ⚠️ نیاز به نصب `ffmpeg` دارد. در Colab به صورت پیش‌فرض موجود است.

---

### ۵.۲. ثبت داده‌های حرکتی و رسم نمودار

```python
import matplotlib.pyplot as plt

positions_log = []
for i in range(300):
    joint_state = p.getJointState(robotId, 1)  # جوینت ۱
    positions_log.append(joint_state[0])  # موقعیت
    p.stepSimulation()

# رسم نمودار
plt.figure(figsize=(10, 4))
plt.plot(positions_log, label="موقعیت جوینت ۱")
plt.title("حرکت جوینت در طول زمان")
plt.xlabel("فریم")
plt.ylabel("موقعیت (رادیان)")
plt.legend()
plt.grid(True)
plt.show()
```

---

## 🚀 بخش ۶: کاربردهای پیشرفته

### ۶.۱. شبیه‌سازی ربات‌های دوپا

```python
# بارگذاری ربات انسان‌نما
humanoidId = p.loadURDF("humanoid/humanoid.urdf", [0, 0, 1])
```

> ✅ این ربات‌ها برای شبیه‌سازی راه‌رفتن و تعادل مناسب هستند.

---

### ۶.۲. استفاده در یادگیری تقویتی (RL)

PyBullet با محیط‌های `gym` سازگار است:

```python
import gym
import pybullet_envs  # خودکار محیط‌های pybullet را ثبت می‌کند

env = gym.make("Ant-v4", render=True)
obs = env.reset()
for i in range(1000):
    action = env.action_space.sample()  # حرکت تصادفی
    obs, reward, done, info = env.step(action)
    if done:
        obs = env.reset()
env.close()
```

> 📌 پروژه پیشنهادی: آموزش یک عامل RL برای راه‌رفتن ربات Ant با استفاده از `stable-baselines3`.

---

### ۶.۳. شبیه‌سازی محیط‌های سفارشی

می‌توانید محیط‌های خود را با فرمت `URDF` یا `SDF` بسازید:

```python
custom_env_id = p.loadURDF("my_custom_room.urdf")
```

> 💡 نرم‌افزارهایی مثل **Blender + Phyronet** یا **SolidWorks + URDF Exporter** به ساخت مدل کمک می‌کنند.

---

## 🧰 بخش ۷: رفع خطاهای رایج

| مشکل | علت احتمالی | راه‌حل |
|------|-------------|--------|
| صفحه سیاه در Colab | عدم پشتیبانی از GUI | از `pyvirtualdisplay` استفاده کنید |
| ربات می‌افتد | گرانش فعال است و پایه ثابت نیست | `useFixedBase=True` را اضافه کنید |
| جوینت‌ها حرکت نمی‌کنند | کنترل موتور تنظیم نشده | `setJointMotorControl2` را فراموش نکنید |
| تصویر دوربین خالی است | دوربین خارج از صحنه است | موقعیت دوربین را تنظیم کنید |

### 🔹 استفاده از `pyvirtualdisplay` در Google Colab

```python
# فقط در Colab اجرا شود
!apt-get install -y xvfb
!pip install pyvirtualdisplay

from pyvirtualdisplay import Display
display = Display(visible=0, size=(1400, 900))
display.start()
```

سپس شبیه‌سازی را با `p.DIRECT` اجرا کنید:

```python
p.connect(p.DIRECT)  # بدون نمایش
```

---

## 📁 بخش ۸: پروژه‌های پیشنهادی برای دانشجویان

| پروژه | توضیح |
|-------|--------|
| **۱. ربات با هدف تعیین موقعیت** | ربات را به یک نقطه در فضا برسانید و مسیر حرکت را ثبت کنید. |
| **۲. ربات چهارپا (MiniCheetah)** | حرکت راه‌رفتن را شبیه‌سازی کنید و تعادل را بررسی کنید. |
| **۳. ربات با دوربین و تشخیص شیء** | تصویر بگیرید و موقعیت یک جعبه را تشخیص دهید. |
| **۴. شبیه‌سازی مسابقه رباتیک** | دو ربات در یک محیط با هم تعامل داشته باشند (مثلاً هل دادن). |
| **۵. یادگیری تقویتی برای ربات بازویی** | با `stable-baselines3` یک عامل RL برای بلند کردن جعبه آموزش دهید. |

---

## 📚 منابع و مراجع

- 📄 [مستندات رسمی PyBullet](https://docs.google.com/document/d/10sXEhzFRSnvFcl3XxNGhnD4N2SedytjwJMmPFpx44E4/pub)
- 🔗 [ریپازیتوری GitHub](https://github.com/bulletphysics/bullet3)
- 🎥 [آموزش‌های ویدیویی (YouTube)](https://www.youtube.com/results?search_query=pybullet+tutorial)
- 🧪 [PyBullet Gym Envs](https://github.com/benelot/pybullet-gym)
- 📘 [کتاب "Programming a Robotic Car" — شبیه‌سازی فیزیکی](https://www.coursera.org/learn/cs373)

---

## 🙌 نتیجه‌گیری

PyBullet یک ابزار قدرتمند، رایگان و انعطاف‌پذیر برای شبیه‌سازی سیستم‌های فیزیکی و رباتیک است. با استفاده از آن، دانشجویان می‌توانند:

- مفاهیم کنترل و دینامیک را در عمل ببینند.
- الگوریتم‌های پیچیده هوش مصنوعی را تست کنند.
- پروژه‌های تحقیقاتی واقع‌گرایانه اجرا کنند.

> 🎯 این آموزش فقط شروع راه است. دنیای شبیه‌سازی در انتظار شماست!

---

## 📎 پیوست: نصب و اجرا در Google Colab

برای اجرای PyBullet در Google Colab:

```python
# مرحله ۱: نصب
!pip install pybullet pyvirtualdisplay

# مرحله ۲: راه‌اندازی نمایش مجازی
from pyvirtualdisplay import Display
display = Display(visible=0, size=(1400, 900))
display.start()

# مرحله ۳: اجرای شبیه‌سازی در حالت بدون نمایش
import pybullet as p
p.connect(p.DIRECT)  # بدون GUI
```

> ✅ حالا می‌توانید شبیه‌سازی را اجرا کنید و خروجی را ضبط یا نمایش دهید.

---

> ✍️ تهیه شده توسط: [نام شما]  
> 📅 تاریخ: اسفند ۱۴۰۲  
> 🎓 برای درس رباتیک / هوش مصنوعی / شبیه‌سازی سیستم‌های کنترل
```

---

## ✅ نحوه استفاده

1. این متن را در یک ویرایشگر متن (مثل VS Code، Notepad++ یا Google Docs) کپی کنید.
2. با نام `pybullet_tutorial.md` ذخیره کنید.
3. به دانشجویان بدهید یا در سیستم آموزشی آپلود کنید.

اگر مایل باشید، می‌توانم این فایل را به صورت **PDF** یا **ZIP از یک پروژه کامل** (با فایل‌های اجرایی) نیز ارائه دهم. فقط بگویید!

https://pybullet.org/